# Continuous-time Markov processes

A **Markov process** is a stochastic model for a system, in which the
evolution of the system is determined only by the *current* state of the
system, not by any state in the past *(except perhaps through
derivatives of the dynamical quantities, where those derivatives exist -
often they do not)*. Such a system may be said to possess the **Markov
property** ([Wikipedia](https://en.wikipedia.org/wiki/Markov_property)),
or to be ‘memoryless’.

The most famous and elementary types of Markov processes are Markov
chains ([Wikipedia](https://en.wikipedia.org/wiki/Markov_chain)), which
have a discrete set of states (which may be infinite) and evolution
between the states takes place in discrete, uniform units of time.

However, for some systems a **continuous-time Markov process** may be
more appropriate. Under continuous-time evolution, we speak not of
‘transition probabilities’ but rather of ‘transition rates’. This is
most important when we care about the dependence of the dynamical
properties (*e.g.* the population of a species) on time, and when the
rate of change depends on the quantity (*e.g.* higher population implies
shorter periods between birth or death events).

The CTMP examples below have discrete states but continuous time. This
distinguishes them from Brownian processes, where the dynamical
quantities (*e.g.* the position of a particle) are also continuous.

At each state, the transition rates to other possible states are given
by specifying a transition function, and typically these transitions are
assumed to be Poisson processes: each transition has a rate of occuring
per unit time, and the first event to ‘trigger’ is the transition that
is actually taken. The time to transition is therefore distributed
according to an exponential distribution.

In [ ]:
import logging
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng

from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

logging.getLogger().setLevel(logging.WARN)

One way to numerically simulate a CTMP is to use an *event-driven
simulation*. At a given state, each possible transition has an
associated rate. For each transition, a random exponential variable with
that rate is sampled: this is the time until that transition would
trigger in the absence of any others. The smallest of these transition
times determines the transition that is actually taken.

To avoid numerical overflow or underflow, we terminate the solution if
transition rates become too large (and therefore transition times are
too small).

In [ ]:
from ctmp_simulation import evolution

## Immigration-death system (Mathematical Biology Sheet 4 Question 1)

Rates: \* $\lambda_1, \lambda_2$ - birth of one or two offspring \*
$\beta n$ - deaths

Without loss of generality, we can take $\beta = 1$ and rescale time
accordingly.

In [ ]:
@interact(
    l_1=widgets.FloatSlider(min=0, max=10, value=5, continuous_update=False),
    l_2=widgets.FloatSlider(min=0, max=10, value=5, continuous_update=False),
    beta=widgets.FloatSlider(min=0.1, max=2, value=1, continuous_update=False),
)
def immigration_death_demo(l_1, l_2, beta):
    def transition_fun(t, s, l_1, l_2, beta):
        return {
            s + 1: l_1,
            s + 2: l_2,
            s - 1: beta * s
        }

    tmax = 50
    ts, history = evolution(transition_fun, (0, tmax), 0,
                        args=(l_1, l_2, beta),
                        # t_eval=np.arange(0, tmax, 1)
                        )

    mean = 1/beta * (l_1 + 2*l_2)
    sigma = (1/beta * (l_1 + 3*l_2))**0.5
    plt.plot(ts, history, 'r-',
             ts, [mean] * len(ts), 'k--',
             ts, [mean + sigma] * len(ts), 'k:',
             ts, [mean - sigma] * len(ts), 'k:',
    )
    plt.title(f'$\\mu = {mean:.2f}$, $\\sigma^2 = {sigma**2:.2f}$')
    plt.show()

## Another immigration-death system (Mathematical Biology Sheet 4 Question 2)

Similar to the above, except now we introduce two or three, rather than
one or two, at a time.

In [ ]:
@interact(
    l_2=widgets.FloatSlider(min=0, max=12.5, value=5, continuous_update=False),
    l_3=widgets.FloatSlider(min=0, max=10, value=5, continuous_update=False),
    beta=widgets.FloatSlider(min=0, max=2, value=1, continuous_update=False),
)
def immigration_2_or_3_death_demo(l_2, l_3, beta):
    def transition_fun(t, s, l_2, l_3, beta):
        return {
            s + 2: l_2,
            s + 3: l_3,
            s - 1: beta * s
        }

    tmax = 500
    ts, history = evolution(transition_fun, (0, tmax), 0,
                        args=(l_2, l_3, beta),
                        # t_eval=np.arange(0, tmax, 1)
                        )

    mean = 1/beta * (2*l_2 + 3*l_3)
#     sigma = 0  # FIXME
    sigma = ((3 * mean - 5 * l_2 / beta) / 3) ** 0.5
    plt.plot(ts, history, 'r-',
             ts, [mean] * len(ts), 'k--',
             ts, [mean + sigma] * len(ts), 'k:',
             ts, [mean - sigma] * len(ts), 'k:',
    )
    plt.title(f'$\\mu = {mean:.2f}$, $\\sigma^2 = {sigma**2:.2f}$')
    plt.show()

## Immigration-death system with quadratic death rate (Mathematical Biology sheet 4 question 3)

Throughout we normalize such that $\beta = 1$, so that we can work with
$r = \lambda / \beta$ instead.

In the steady state, the generating function satisfies the equation $$
s \frac{\partial^2\phi}{\partial s^2} = r \phi, \qquad \phi(s=1) = 1, \text{with $\phi(s=0)$ bounded}.
$$ This equation does not have a simple closed-form solution, although
the solution can be written as: $$
\phi = \frac{ s^{1/2} I_1\left(2(rs)^{1/2}\right) }{I_1\left(2r^{1/2}\right)}
$$ where $I_n$ is a [modified Bessel function of the first
kind](https://mathworld.wolfram.com/ModifiedBesselFunctionoftheFirstKind.html),
the solution to the differential equation $$
x^2 y'' + x y' - (x^2 + n^2) y = 0
$$ that is regular at $x = 0$. Then (according to Wolfram Alpha)… $$
\mu = \frac{\partial\phi}{\partial s}(s=1) = \frac{1}{2} \left[
\frac{r^{1/2}}{I_1(2 r^{1/2})} \left(
I_0(2r^{1/2}) + I_2(2r^{1/2})
\right) + 1\right].
$$

We also know that $$
\sigma = \sqrt{r + \mu - \mu^2}.
$$

It can be shown ([source](https://dlmf.nist.gov/10.40)) that, as
$z \rightarrow \infty$, then for any $n$, $$
I_n(z) \sim \frac{\mathrm{e}^z}{(2\pi z)^{1/2}},
$$ and therefore (at leading order):
$$ \mu \sim r^{1/2}, \quad \sigma \sim r^{1/4}. $$ This is consistent
with what you might expect from a scaling (dimensional analytical)
argument.

In [ ]:
# modified Bessel function of the first kind
from scipy.special import iv
from tqdm.auto import tqdm

@interact(
    r=widgets.FloatSlider(min=0, max=200, value=100, step=0.1, continuous_update=False),
)
def immigration_nonlinear_death_demo(r):
    def f(s):
        return s * (s - 1)

    def transition_fun(t, s, r):
        return {
            s + 1: r,
            s - 1: f(s)
        }

    n0 = 0

    tmax = 10
    ts, history = evolution(transition_fun, (0, tmax), n0,
                        args=(r,),
                        # t_eval=np.arange(0, tmax, 1)
                        )

    z = 2*r**0.5
    mean = 0.5 * (r**0.5/iv(1, z) * (iv(0, z) + iv(2, z)) + 1 )
    sigma = (r + mean - mean**2) ** 0.5

    fig, axs = plt.subplots(2, 2, figsize=(12, 7))
    ax = axs[0, 0]
    ax.plot(ts, history, 'r-',
             ts, [mean] * len(ts), 'k--',
             ts, [mean + sigma] * len(ts), 'k:',
             ts, [mean - sigma] * len(ts), 'k:',
    )

    ax = axs[1, 0]
    ax.hist(history[:-1], weights=np.diff(ts), density=True)
    ax.set_xlim([0, 20])

    nsims = 100
    tmax = 0.5
    t_eval = np.arange(0, tmax, 0.001)
    ax = axs[0, 1]
    finals = []
    histories = []
    for i in tqdm(range(nsims)):
        ts, history = evolution(transition_fun, (0, tmax), n0,
                        args=(r,),
                        t_eval=t_eval)
        if i < 10:
            ax.plot(ts, history,
                lw=0.5
            )
        histories.append(history)
        finals.append(history[-1])

    ax.plot(
        ts, [mean] * len(ts), 'k--',
        ts, [mean + sigma] * len(ts), 'k:',
        ts, [mean - sigma] * len(ts), 'k:',
    )

    histories = np.array(histories)
    mu = np.mean(histories, axis=0)
    std = np.std(histories, axis=0)

    ax.plot(t_eval, mu, 'k', lw=2)
    ax.plot(t_eval, mu + std, 'k--', lw=1)
    ax.plot(t_eval, mu - std, 'k--', lw=1)
    ax.fill_between(t_eval, mu - std, mu + std, fc='k', alpha=0.4)

    ax = axs[1, 1]
    ax.hist(finals, density=True)
    ax.set_xlim([0, 20])

    plt.show()

## Logistic dynamics

Here is another “logistic dynamics” system for another population of
size $n(t)$. The birth rate is $rn$ and the death rate is $-n^2$. The
equilibrium population size is therefore $r$.

In [ ]:
@interact(
    r=widgets.FloatSlider(min=1, max=100, value=30, step=1, continuous_update=False),
)
def logistic_demo(r):
    def transition_fun(t, n, r):
        return {
            n + 1: r*n,
            n - 1: n**2
        }

    tmax = 3
    ts, history = evolution(transition_fun, (0, tmax), state0=1,
                        args=(r,))

    plt.plot(ts, history, 'r-')
    plt.show()

In [ ]:
@interact(
    r=widgets.FloatSlider(min=1, max=100, value=30, step=1, continuous_update=False),
)
def logistic_demo(r):
    def transition_fun(t, n, r):
        return {
            n + 1: r*n,
            n - 1: n**2
        }

    tmax = 3
    ts, history = evolution(transition_fun, (0, tmax), state0=1,
                        args=(r,),
                        t_eval = np.linspace(0, tmax, 50)
                        )

    plt.plot(ts, history, 'r-')
    plt.show()

## Stochastic Lotka-Volterra (two-population system)

The [Lotka-Volterra
equations](https://en.wikipedia.org/wiki/Lotka%E2%80%93Volterra_equations)
are a family of simple models for the populations of a predator-prey
system. The original model is a deterministic dynamical system,
consisting of a pair of ordinary differential equations (continuous
time, continuous populations). This model is suitable for large
populations but leads to the ‘atto-fox’ problem: populations never
become extinct but reach extremely small quantities that are somehow
able to bounce back. One resolution is to turn the populations into
discrete quantities (you can’t have half a fox, or $10^{-18}$ foxes –
but you can rescale units to talk in terms of 1000s of foxes, for
example). Having done that we can also add uncertainty by introducing a
transition function between states.

In [ ]:
def lv_transition_fun(t, n):
    x, y = n
    s = .001
    return {
        (x + s, y): 2/3 * x,
        # (x - s, y): x * x / 20,
        (x - 4/3*s, y+s): x * y,
        (x, y - s): y
    }

In [ ]:
@interact()
def lv_demo():
    ts, history = evolution(
        lv_transition_fun,
        (0, 50_000),
        [1, 0.75],
        t_eval=np.linspace(0, 50_000, 1001),
        maxrate=10,
    )
    prey, predators = history[:, 0], history[:, 1]
    fig = plt.figure(figsize=(8, 3))
    plt.plot(ts, prey, 'g-',
             ts, predators, 'r-')
    plt.grid()
    ax = plt.gca()
    ax.set_ylim([0, None])
    ax.legend(['prey', 'predators'])
    plt.show()

    fig = plt.figure(figsize=(15, 4))
    plt.plot(prey, predators, 'k')
    plt.grid()
    ax = plt.gca()
    ax.set_aspect('equal')
    ax.set_xlabel('prey')
    ax.set_ylabel('predators')
    ax.set_xlim([0, None])
    ax.set_ylim([0, None])
    plt.show()

While the basic, deterministic Lotka-Volterra system has solutions that
are closed loops, the realisations of the stochastic system also exhibit
periodic-like behaviour, but with some variation and therefore not
closed loops.

If we calculate the *ensemble average* of several realisations (perhaps
with different initial conditions), we will find that the average also
executes periodic behaviour. However, the variance (uncertainty) grows
over time and eventually comes to dominate over the periodic behaviour.

In [ ]:
from tqdm.auto import tqdm
@interact()
def lv_ensemble_demo():
    # Same initial conditions
    n_states = 20
    initial_states = [(1, 1) for _ in range(n_states)]

    # # Or different initial conditions
    # initial_states = [(1 + r*np.cos(th), 1 + r*np.sin(th))
    #                   for r in np.linspace(.1, .4, 4)
    #                   for th in np.linspace(0, 2*np.pi, 9)[:-1]]

    tmax = 80_000
    ts = np.linspace(0, tmax, 201)

    results = [evolution(lv_transition_fun, (0, tmax), s, t_eval=ts,
                          maxrate=20) for s in tqdm(initial_states)]
    histories = np.array([r.history for r in results])
    xs = histories[:, :, 0]
    ys = histories[:, :, 1]

    # Ensemble averages and stdevs
    average_x = np.average(xs, axis=0)
    x_std = np.std(xs, axis=0)
    average_y = np.average(ys, axis=0)
    y_std = np.std(ys, axis=0)

    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    ax.plot(ts, average_x, 'g-', label="prey")
    ax.plot(ts, average_x - x_std, 'g--')
    ax.plot(ts, average_x + x_std, 'g--')
    ax.fill_between(
        ts, average_x - x_std, average_x + x_std, fc='g', alpha=0.3
    )
    ax.plot(ts, average_y, 'r-', label='predators')
    ax.plot(ts, average_y - y_std, 'r--')
    ax.plot(ts, average_y + y_std, 'r--')
    ax.fill_between(
        ts, average_y - y_std, average_y + y_std, fc='r', alpha=0.3
    )
    ax.legend()

    @interact(step=widgets.IntSlider(min=0, max=len(ts)-1, value=0, continuous_update=False))
    def plot_evolution(step):
        final_states = [h[1, :] for h in histories]
        plt.figure(figsize=(4, 4))
        plt.plot(
            average_x, average_y, 'r-',
            [x[step][0] for x in histories],
            [x[step][1] for x in histories],
            'k.',
            average_x[step], average_y[step], 'ro'
        )
        plt.gca().set_xlim([-.1, 2])
        plt.gca().set_ylim([-.1, 2])
        plt.gca().set_aspect('equal')
        plt.gca().grid()

## Stochastic harmonic oscillator

Here is a simpler examople of a 2D random walk process that is somewhat
analogous to the simple harmonic oscillator. The mean flow should obey
the same equations as the simple harmonic oscillator (in 2D phase
space).

In [ ]:
def sho_transition_fun(t, s):
    x, y = s
    return {
        (x + 1, y): y,
        (x - 1, y): -y,
        (x, y + 1): -x,
        (x, y - 1): x
    }

In [ ]:
@interact()
def sho_demo():
    tmax = 100
    ts, history = evolution(sho_transition_fun, (0, tmax), [40, 40],
                        t_eval=np.linspace(0, tmax, 121))

    fig, axs = plt.subplots(2, 1)
    ax = axs[0]
    ax.plot(ts, history)
    
    ax = axs[1]
    ax.plot(history[:, 0], history[:, 1])
    plt.show()

Each individual realisation is noisy. The ensemble average again shows
remarkable periodicity but there is plenty of variance as time
progresses.

In [ ]:
@interact()  # no parameters, just for namespacing
def sho_ensemble_demo():
    # initial_states = [(30 + r*np.cos(th), 40 + r*np.sin(th))
    #               for r in np.linspace(1, 5, 5)
    #               for th in np.linspace(0, 2*np.pi, 9)]
    # All trajectories start off the same
    initial_states = [(50, 0) for _ in range(10)]
    nsim = len(initial_states)

    t_eval = np.linspace(0, 25, 161)
    results = [
        evolution(sho_transition_fun, (0, 25), s, t_eval=t_eval)
        for s in initial_states
    ]
    histories = np.array([r.history for r in results])
    print(histories.shape)

    xs = histories[:, :, 0]
    ys = histories[:, :, 1]

    # ensemble averages and standard deviations
    average_x = np.average(xs, axis=0)
    x_std = np.std(xs, axis=0)
    average_y = np.average(ys, axis=0)
    y_std = np.std(ys, axis=0)

    fig, axs = plt.subplots(2, 1, figsize=(6, 3))
    ax = axs[0]
    ax.plot(t_eval, average_x, 'g-')
    ax.plot(t_eval, average_x - x_std, 'g--')
    ax.plot(t_eval, average_x + x_std, 'g--')
    ax = axs[1]
    ax.plot(t_eval, average_y, 'b-')
    ax.plot(t_eval, average_y - y_std, 'b--')
    ax.plot(t_eval, average_y + y_std, 'b--')

    @interact(step=widgets.IntSlider(min=0, max=len(histories[0])-1, value=0, continuous_update=False))
    def plot_evolution(step):
        final_states = [h[step][1] for h in histories]

        plt.figure(figsize=(4, 4))
        plt.plot(
            average_x, average_y, 'r-',
            average_x[step], average_y[step], 'ro',
            [x[step, 0] for x in histories],
            [x[step, 1] for x in histories],
                'k.',
            )
        plt.gca().set_xlim([-60, 60])
        plt.gca().set_ylim([-60, 60])
        plt.gca().set_aspect('equal')
        plt.gca().grid()

## Decay with regeneration

In [ ]:
def transition_fun(t, s):
    x, y = s
    return {
        (x + 1, y): 1 - 0.05*x + y,
        (x - 1, y): 1 + 0.05*x - y,
        (x, y + 1): 1 - 0.05*y - x,
        (x, y - 1): 1 + 0.05*y + x
    }

ts, history = evolution(transition_fun, (0, 100), [10, 10],
                    t_eval=np.linspace(0, 100, 151))

plt.figure(figsize=(14, 5))
plt.plot(ts, history)
plt.show()

plt.plot(history[:, 0], history[:, 1])
plt.show()

In [ ]:
# Warning: This is slow.

initial_states = [(60 + r*np.cos(th), 60 + r*np.sin(th))
                  for r in np.linspace(1, 5, 5)
                  for th in np.linspace(0, 2*np.pi, 9)[:-1]]


results = [
    evolution(transition_fun, (0, 100), s, t_eval=np.linspace(0, 100, 251)) 
    for s in tqdm(initial_states)
]
histories = [r.history for r in results]
average_x = np.average([h[:, 0] for h in histories], axis=0)
average_y = np.average([h[:, 1] for h in histories], axis=0)

In [ ]:
@interact(step=widgets.IntSlider(min=0, max=len(histories[0])-1, value=0, continuous_update=False))
def plot_evolution(step):
    final_states = [h[step][1] for h in histories]

    plt.figure(figsize=[8,8])
    plt.plot(
        average_x, average_y, 'r-',
        [x[step, 0] for x in histories],
        [x[step, 1] for x in histories],
            'k.',
        average_x[step], average_y[step], 'ro',
    )
    plt.gca().set_xlim([-120, 120])
    plt.gca().set_ylim([-120, 120])
    plt.gca().set_aspect('equal')
    plt.gca().grid()

## A singular process

This is a random walk forwards where the rate of advancing from $s$ to
$s + 1$ is equal to $s^2$ (capped at an arbitrary maximum value of
$s = 1000$).

This is the stochastic version of the dynamical system $$
\dot{s} = s^2,
$$

which is a classic example of a system with a ‘finite-time singularity’:
its solutions $$ 
s = \frac{1}{s_0^{-1} - t}
$$

become infinite in a finite amount of time $t = s_0^{-1}$.

Contrast this to $\dot{s} = s$, with solution $s = s_0 \exp t$, which
diverges but only as $t \rightarrow \infty$.

The stochastic version also exhibits singular behaviour but there is a
lot more uncertainty over where precisely the blowup occurs.

In [ ]:
@interact()
def singular_demo():
    def transition_fun(t, s):
        if s ** 2 <= 1e6:
            return {
                s + 1: s ** 2,
            }
        return {}

    s0 = 1
    ts, history = evolution(transition_fun, (0, 2), s0,
                       maxrate=1e6)

    fig, ax = plt.subplots(1, 1)

    ax.plot(ts, history, 'r-', label='stochastic outcome')

    t_eval = np.linspace(0, 2, 100)
    results = [evolution(transition_fun, (0, 2), s0,
                       maxrate=1e6, t_eval=t_eval) 
                       for _ in range(20)]
    histories = [r.history for r in results]
    mu = np.mean(histories, axis=0)
    std = np.std(histories, axis=0)

    ax.plot(t_eval, mu, 'b-', label='ensemble average')
    ax.plot(t_eval, mu-std, 'b--')
    ax.plot(t_eval, mu+std, 'b--')
    ax.fill_between(t_eval, mu-std, mu+std, fc='b', alpha=0.1)

    t01 = np.linspace(0, 1, 1000)[:-1]
    ax.plot(t01, 1/(1/s0 - t01), 'k--', label='continuous analogue')
    ax.set_yscale('log')
    ax.legend()

    plt.show()

## Queueing process with multiple servers

Inspired when I was living at Minack with 40 other people with three
showers between us.

In [ ]:
@interact(
    in_rate=widgets.FloatSlider(min=0, max=5, value=1, continuous_update=False),
    out_rate=widgets.FloatSlider(min=0, max=5, value=1, continuous_update=False),
    n_servers=widgets.IntSlider(min=0, max=12, value=3),
)
def queueing_process(in_rate, out_rate, n_servers):
    def transition_fun(t, s):
        todo, inprog, done = s
        outcomes = {}
        if inprog == n_servers:  # max capacity already, so new tickets go to the queue
            outcomes[todo + 1, inprog, done] = in_rate
        else:
            outcomes[todo, inprog + 1, done] = in_rate

        if todo > 0:
            outcomes[todo - 1, inprog, done + 1] = out_rate * inprog
        else:
            outcomes[0, inprog - 1, done + 1] = out_rate * inprog


        return outcomes

    ts, history = evolution(transition_fun, (0, 100), (0, 0, 0),
                        t_eval=np.linspace(0, 100, 101))

    ax = plt.gca()
    ax.plot(
        ts, history[:, 0], 'r-',
        ts, history[:, 1], 'k:',
#         ts, history[:, 2], 'k-'
    )
    ax.set_ylim([0, None])
    ax.grid()
    ax.legend(['queueing', 'in progress', 'done'])
#     return ts, history